# Manual Quality Checks and Preprocessing Configuration

**Author:** Noah Mba, noah.mba@fu-berlin.de  
**Date:** July 13, 2026  
**AI Acknowledgements:** Co-authored/Supported by Gemini and Claude 5 Sonnet  

---

This notebook prepares the pre-processing steps that require manual input on a subject-level:

1. Checking the integrity of the recorded EEG data.
2. Selection of channels for interpolation.
3. Running preliminary ICA and selecting independent components for exclusion.

The resulting information is logged in `preprocessing_config.csv`, which is used as input in the subsequent automated preprocessing notebook (`3_preprocessing.ipynb`). This allows the pipeline to loop over a selection (or all) of the participants.

### 1. Setup, Configuration, and Behavioral Exclusion Check

This initialization cell prepares the workspace for the current participant. Specifically, it:

* Imports libraries and enables interactive plotting (`%matplotlib qt`), which is required for manual EEG/ICA inspection.
* Sets the current subject ID and locates the BIDS and derivatives directories.
* Loads the existing `preprocessing_config.csv` metadata table or creates a new one if this is the first run.
* Cross-references the subject with `subject_exclusions.json`. If the participant failed behavioral criteria,     
it automatically logs their exclusion in `preprocessing_config.csv` and throws a warning so we know to skip the rest of the notebook.

In [53]:
import mne
import pandas as pd
import json
from pathlib import Path
from mne_bids import BIDSPath, read_raw_bids
import matplotlib
import matplotlib.pyplot as plt
from mne_icalabel import label_components
import os

# Sets the matplotlib backend to open a separate interactive window
%matplotlib qt 

# ==========================================
# 1.1. DEFINE CURRENT SUBJECT
# ==========================================
# Change this ID for every participant you want to inspect
subj = "21"

# ==========================================
# 1.2. DEFINE TIME BUFFER FOR THE CROPPING IN 2.1B
# ==========================================

BUFFER_SEC = 20  # margin kept before/after each phase

# ==========================================
# 1.3. DEFINE PATHS & LOAD EXCLUSIONS DUE TO BEHAVIORAL RESULTS
# ==========================================
# This notebook should be located in project_folder/scripts/eeg
project_root = Path.cwd().parent.parent
bids_root = project_root / "data" / "bids"
derivatives_dir = project_root / "data" / "derivatives"

# Load json object with subjects that should be excluded due to poor behavioral performance
exclusion_file = derivatives_dir / "subject_exclusions.json"

# The JSON object is loaded into exclusions, a Python dictionary
with open(exclusion_file, "r") as f:
    exclusions = json.load(f)

# Use .get() to safely load the list of values, defaulting to an empty list if the key is missing
behavioral_exclusions = exclusions.get("behavioral_exclusions", [])

print("The following subjects are marked for exclusion based on behavioral criteria:")
print(behavioral_exclusions)

# The configuration file should live in the derivatives directory
config_path = derivatives_dir / "preprocessing_config.csv"

# ==========================================
# 1.4. LOAD OR CREATE CONFIGURATION FILE
# ==========================================
if config_path.exists():
    df_config = pd.read_csv(config_path, dtype={'subject': str})
    print(f"\nExisting config loaded. {len(df_config)} subjects inspected so far.")
else:
    df_config = pd.DataFrame(columns=[
        "subject", 
        "is_excluded", 
        "eeg_enc_recorded", 
        "eeg_ret_recorded", 
        "beh_enc_complete", 
        "beh_ret_complete",
        "bad_channels", 
        "bad_icas", 
        "notes_exclusion", 
        "notes_beh", 
        "notes_eeg"
    ])
    print("\nNo existing config found. Creating a new detailed metadata table.")

# ==========================================
# 1.5. SAFETY CHECK & AUTO-LOGGING
# ==========================================
# Check if the current subject is in the exclusion list
subject_is_excluded = subj in behavioral_exclusions

if subject_is_excluded:
    # 4a. Throw a highly visible warning
    print("\n" + "!"*65)
    print(f" WARNING: Subject {subj} is in the behavioral_exclusions list!")
    print("!"*65)
    
    # 4b. Automatically log this into the config dataframe
    # NOTE: since ICA/channel inspection never runs for auto-excluded subjects,
    # bad_channels/bad_icas/notes_eeg are left empty here.
    auto_notes = "Auto-excluded: Behavioral criteria not met."
    
    if subj in df_config['subject'].values: # Checking if the current subject already exists in df_config table
        idx = df_config.index[df_config['subject'] == subj].tolist()[0] # Retrieves the row in df_config for that subject
        df_config.loc[idx, 'is_excluded'] = True # Sets is_excluded column to True for that subject
        df_config.loc[idx, 'notes_exclusion'] = auto_notes # Adds the exclusion note for that subject
    else:
        new_row = pd.DataFrame([{
            "subject": subj, 
            "is_excluded": True,
            "eeg_enc_recorded": pd.NA,
            "eeg_ret_recorded": pd.NA,
            "beh_enc_complete": pd.NA,
            "beh_ret_complete": pd.NA,
            "bad_channels": "", 
            "bad_icas": "",
            "ica_notes": "",
            "notes_exclusion": auto_notes,
            "notes_beh": "",
            "notes_eeg": ""
        }])
        df_config = pd.concat([df_config, new_row], ignore_index=True)
        
    # 4c. Save immediately
    df_config.to_csv(config_path, index=False)
    print(f"\n--> Action taken: Subject {subj} automatically marked as excluded in the config.")
    print("--> You can SKIP the rest of the cells in this notebook for this subject.")

else:
    print(f"\n--> Subject {subj} is clear for preprocessing. Proceed to the next cells.")

The following subjects are marked for exclusion based on behavioral criteria:
['05', '06', '07', '08', '13', '14', '16', '18', '19', '24', '25', '28', '31', '34', '39', '40']

Existing config loaded. 2 subjects inspected so far.

--> Subject 21 is clear for preprocessing. Proceed to the next cells.


### 2. Checking the integrity of the recorded EEG data

The following two plotting cells allow me to assess the completeness of the EEG recordings:

* Have triggers been successfully recorded and mapped as expected?
* Do we have signal recordings from 64 channels throughout the whole experimental procedure?

#### 2.1 Event markers plot
This plot shows us whether triggers for both phases were recorded completely.

In [54]:
# ==========================================
# 2.1 PLOT EVENT MARKERS 
# ==========================================

# Set BIDS path
bids_path = BIDSPath(
    subject=subj, 
    task='loc', 
    datatype='eeg', 
    root=bids_root)

# Load Raw
raw_eeg, bids_event_id = read_raw_bids(
    bids_path=bids_path, 
    return_event_dict=True,  # Extracts events mapping from BIDS (previous botebook)
    verbose='error'
)
raw_eeg.load_data()

# Load events
events, event_id = mne.events_from_annotations(
    raw_eeg, 
    event_id=bids_event_id,  
    verbose=False
)

# Plot event markers as function of experiment run time
fig_ev = mne.viz.plot_events(
    events, 
    sfreq=raw_eeg.info["sfreq"], 
    first_samp=raw_eeg.first_samp, 
    event_id=event_id
)

Reading 0 ... 2767979  =      0.000 ...  2767.979 secs...


C:\Users\noahm\AppData\Local\Temp\ipykernel_6316\1901495321.py:28: RuntimeWarning: More events than default colors available. You should pass a list of unique colors.
  fig_ev = mne.viz.plot_events(
c:\Users\noahm\mne-python\1.11.0_0\Lib\site-packages\mne\viz\utils.py:160: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  (fig or plt).show(**kwargs)


#### 2.1b Cropping the continuous data

We decided to record the whole experimental session, including the instruction, breaks, and distractor phase, which are of no interest for our EEG analyses and increase the computational demands of our subsequent processing steps, while simultaneously introducing noise. Thus, we crop the EEG data around the start and end of the encoding and retrieval phase, respectively. Then we concatenate the two parts together again, while adding an empty buffer to minimize edge-detection artefacts during the filtering later. This way, bad-channel detection and ICA still run once
per subject on a single Raw object, which makes sense to not introduce condition-specific differences in the preprocessing parameters.

This will make it easier to open the interactive MNE plot of the continuous data in this notebook. The same cropping procedure will also be performed as part of the preprocessing pipeline in Notebook 3.

In [55]:
def get_onset(raw, description):
    """Get the onset (in seconds) of the first annotation matching `description`."""
    onset = raw.annotations.onset[raw.annotations.description == description]
    if len(onset) == 0:
        raise ValueError(f"No annotation '{description}' found for subject {subj}")
    return onset[0]

enc_start = get_onset(raw_eeg, "enc_start")
enc_end   = get_onset(raw_eeg, "enc_end")
ret_start = get_onset(raw_eeg, "ret_start")
ret_end   = get_onset(raw_eeg, "ret_end")

# Clip buffers to stay within the recording bounds
t_min, t_max = raw_eeg.times[0], raw_eeg.times[-1]
enc_tmin = max(enc_start - BUFFER_SEC, t_min)       # BUFFER_SEC is defined in the configuration cell (#1)
enc_tmax = min(enc_end + BUFFER_SEC, t_max)
ret_tmin = max(ret_start - BUFFER_SEC, t_min)
ret_tmax = min(ret_end + BUFFER_SEC, t_max)

raw_enc = raw_eeg.copy().crop(tmin=enc_tmin, tmax=enc_tmax)
raw_ret = raw_eeg.copy().crop(tmin=ret_tmin, tmax=ret_tmax)

# Concatenate — this inserts a boundary annotation at the seam automatically,
# so later steps (ICA fit, epoching) will not treat the join as continuous signal
raw_eeg = mne.concatenate_raws([raw_enc, raw_ret])

print(f"Cropped subject {subj}: "
      f"encoding [{enc_tmin:.1f}, {enc_tmax:.1f}]s, "
      f"retrieval [{ret_tmin:.1f}, {ret_tmax:.1f}]s "
      f"→ total duration {raw_eeg.times[-1]:.1f}s")

# Optional sanity check: re-plot events on the cropped/concatenated data
events, event_id = mne.events_from_annotations(raw_eeg, event_id=bids_event_id, verbose=False)
fig_ev_cropped = mne.viz.plot_events(
    events, sfreq=raw_eeg.info["sfreq"], first_samp=raw_eeg.first_samp, event_id=event_id
)

Cropped subject 21: encoding [217.6, 898.9]s, retrieval [1631.2, 2768.0]s → total duration 1818.1s


C:\Users\noahm\AppData\Local\Temp\ipykernel_6316\207491924.py:34: RuntimeWarning: More events than default colors available. You should pass a list of unique colors.
  fig_ev_cropped = mne.viz.plot_events(
c:\Users\noahm\mne-python\1.11.0_0\Lib\site-packages\mne\viz\utils.py:160: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  (fig or plt).show(**kwargs)


#### 2.2. Plotting the raw continuous data
To make visual inspection easier, we'll  create a new copy of the cropped raw data, that is highpass & notch filtered, and downsmpled. Do not mark bad channels in the interactive MNE plot! Instead, declare them in `raw_eeg.info['bads']` after viewing all diagnostic plots.

In [ ]:
print(f"Inspecting subject: {subj}")

# Create a copy with basic filters for easier visual inspection
# (1Hz highpass removes slow drifts, 50 notch removes line noise)
raw_viz = raw_eeg.copy().filter(l_freq=1.0, h_freq=None).notch_filter(50).resample(250)

# Plot time series
raw_viz.plot(
    duration=10, 
    n_channels=32, 
    scalings=dict(eeg=20e-6),
    theme="light" 
)

Inspecting subject: 21
Filtering raw data in 2 contiguous segments
Setting up high-pass filter at 1 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal highpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 1.00
- Lower transition bandwidth: 1.00 Hz (-6 dB cutoff frequency: 0.50 Hz)
- Filter length: 3301 samples (3.301 s)

Filtering raw data in 2 contiguous segments
Setting up band-stop filter from 49 - 51 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandstop filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 49.38
- Lower transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 49.12 Hz)
- Upper passband edge: 50.62 Hz
- Upper transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 50.88 Hz)
- F

<mne_qt_browser._pg_figure.MNEQtBrowser(0x1bf532dee70) at 0x000001C07913F0C0>

Channels marked as bad:
none


### 2. Selection of channels for interpolation.

While some bad channels might have been identified during the recording or by viewing the continuous, we will now create some more diagnostic plots to select channels that will need to be interpolated. We will do this by looking at `raw_inspect`, which is a filtered copy of the raw data.

#### 2.1  Power Spectrum Density (PSD) plot
The PSD plot tells us the **power**  of each channel in logarithmic scale (y-axis) across a continuous range of frequencies (x-axis). The underlyig computation is a **Fourier transformation**, in which each channel's time-series data is decomposed into sine waves of varying frequencies. In this plot, the power refers to the squared amplitude of those sine waves at a given frequency.  

This plot can help detect channels that require interpolation. Normal data will show a 1/f decay curve and an alpha peak around 8 to 12 Hz. Red flags:
(a) Near-zero variance across all frequencies: no signal recorded, defect electrode
(b) Elevated power across most/all frequencies relevate to all other channels: muscular activity (EMG)

In [43]:
# Create filtered copy of raw data
raw_inspect = raw_eeg.copy().filter(l_freq=1.0, h_freq=None).notch_filter(50)

Filtering raw data in 2 contiguous segments
Setting up high-pass filter at 1 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal highpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 1.00
- Lower transition bandwidth: 1.00 Hz (-6 dB cutoff frequency: 0.50 Hz)
- Filter length: 3301 samples (3.301 s)

Filtering raw data in 2 contiguous segments
Setting up band-stop filter from 49 - 51 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandstop filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 49.38
- Lower transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 49.12 Hz)
- Upper passband edge: 50.62 Hz
- Upper transition bandwidth: 0.50 Hz (-6 dB cutoff frequency: 50.88 Hz)
- Filter length: 6601 samp

In [47]:
# 'fmax' sets x-axis range from 0 to 100 Hz, allowing me to check for line noise before and after filtering

fig1 = raw_eeg.compute_psd(fmax=100, ).plot() # raw data (unfiltered)
fig1.suptitle("Raw EEG (Unfiltered)", fontsize=14, fontweight='bold')

fig2 = raw_inspect.compute_psd(fmax=100).plot() # 1Hz highpass, and 50 Hz notch filter
fig2.suptitle("Filtered EEG (1Hz Highpass + 50Hz Notch)", fontsize=14, fontweight='bold')

Effective window size : 2.048 (s)
Plotting power spectral density (dB=True).


c:\Users\noahm\mne-python\1.11.0_0\Lib\site-packages\mne\viz\utils.py:160: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  (fig or plt).show(**kwargs)


Effective window size : 2.048 (s)
Plotting power spectral density (dB=True).


c:\Users\noahm\mne-python\1.11.0_0\Lib\site-packages\mne\viz\utils.py:160: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  (fig or plt).show(**kwargs)


Text(0.5, 0.98, 'Filtered EEG (1Hz Highpass + 50Hz Notch)')

#### 2.1 Topoplots

The two topographical plots (for the encoding and retrieval phase) depict the average voltage at each channel across a number of events for a set of time points (-0.5, 0, 0.5, 1 seconds relative to fixation cross). We use fixation crosses to yield a high number of epochs.    

Expected behavior: the brain's electrival activity should spread smoothly across the scalp due to volume conduction. Thus, we can flag channels if they stand out as isolated hot/cold spots relative to their neighbors. 

We will computy the plots on `raw_inspect`, because the massive slow drifts in the unfiltered raw data blows out the color scale of the topomaps.

In [57]:
# 1. Define both events as list of tuples: (name, id)
target_events = [
    ('enc_fixation', 110),
    ('ret_fixation', 210),
]

for event_name, event_id in target_events:

    # 2. Extract events for this specific event only
    events, _ = mne.events_from_annotations(
        raw_inspect,
        event_id={event_name: event_id},
        verbose=False
    )

    # 3. Check if the target ID exists
    if event_id not in events[:, 2]:
        print(f"Warning: Event ID {event_id} ('{event_name}') not found. Skipping.")
        continue

    print(f"Plotting topomap for event: '{event_name}' (ID: {event_id})")

    # 4. Create Epochs
    epochs_viz = mne.Epochs(
        raw_inspect,
        events,
        event_id={event_name: event_id},
        tmin=-0.5, tmax=1,
        baseline=(None, 0),                     # Baseline correction for -0.5 to 0 s
        preload=True,                           # Epoch data is loaded into memory
        reject_by_annotation=True,              # BAD annotations from previous steps are considered
        verbose=False                       
    )

    # 5. Plot Topomap
    fig_topo = epochs_viz.average().plot_topomap(
        show_names=True,
        size=3,
        nrows = 1  # Zur Unterscheidung
    )
    fig_topo.suptitle(f"Topoplots Fixation ({event_name})", fontsize=14, fontweight='bold')
    

Plotting topomap for event: 'enc_fixation' (ID: 110)


c:\Users\noahm\mne-python\1.11.0_0\Lib\site-packages\mne\viz\utils.py:160: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  (fig or plt).show(**kwargs)


Plotting topomap for event: 'ret_fixation' (ID: 210)


c:\Users\noahm\mne-python\1.11.0_0\Lib\site-packages\mne\viz\utils.py:160: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  (fig or plt).show(**kwargs)


### 3. Analyzing ICA results and selecting independent components for exclusion

In the following cells, we run a preliminary Independent Component Analysis (ICA) and save the resulting unmixing matrices into the subjects' derivatives folder. The actual correction of the data based on the ICA computations happens during the batch processing in `3_preprocessing.ipynb`.

ICA is an unsupervised machine learning algorithm that can separate the EEG signal into statistically independent subcomponents that contribute to the recording signal: e.g., neural activity from the brain, eye blinks (EOG), heartbeats (ESC), muscle tension (EMG), channel noise.  

For computational efficiency and susceptibility to low-frequency noise, we run the ICA on an EEG copy that is downsampled, re-referenced to average, and filtered between 1 and 100 Hz. The ICA is then computed using the PICARD algorithm, and using every 3rd data point of the EEG recording.

In [58]:
# ==========================================
# 3. RUN INDEPENDENT COMPONENT ANALYSIS (ICA)
# ==========================================

# Enter bad channels from previous inspection steps
raw_eeg.info['bads'] = ['FT7', 'Fp1', 'AF8']

# We create a filtered, average-referenced, downsampled copy of the raw data for the preliminary ICA
raw_ica = raw_eeg.copy().filter(l_freq=1.0, h_freq=100) 
raw_ica.set_eeg_reference('average')
raw_ica.resample(250)  

# Rank refers to the dimensionality of the data. 64 channels - 1 (average reference) - n bad channels
rank = mne.compute_rank(raw_ica, tol='auto')['eeg']
print(f"Using n_components={rank} based on computed data rank.")

from mne.preprocessing import ICA
ica = ICA(
    n_components=rank, 
    method='picard', # Preconditioned ICA for Real Data: optimization algorithm
    fit_params=dict(ortho=False, extended=True),  # makes picard mathematically equivalent to Extended Infomax, which is widely considered the gold standard
    random_state=97,
    max_iter='auto' 
)
ica.fit(raw_ica, decim=3) # Since we have a longer recording, it's okay to only use every 3rd data point for ICA fitting

Filtering raw data in 2 contiguous segments
Setting up band-pass filter from 1 - 1e+02 Hz

FIR filter parameters
---------------------
Designing a one-pass, zero-phase, non-causal bandpass filter:
- Windowed time-domain design (firwin) method
- Hamming window with 0.0194 passband ripple and 53 dB stopband attenuation
- Lower passband edge: 1.00
- Lower transition bandwidth: 1.00 Hz (-6 dB cutoff frequency: 0.50 Hz)
- Upper passband edge: 100.00 Hz
- Upper transition bandwidth: 25.00 Hz (-6 dB cutoff frequency: 112.50 Hz)
- Filter length: 3301 samples (3.301 s)

EEG channel type selected for re-referencing
Applying average reference.
Applying a custom ('EEG',) reference.
Computing rank from data with rank=None
    Using tolerance 1.5e-10 (2.2e-16 eps * 60 dim * 1.2e+04  max singular value)
    Estimated rank (eeg): 59
    EEG: rank 59 computed from 60 data channels with 0 projectors
Using n_components=59 based on computed data rank.
Fitting ICA to data using 60 channels (please be patie

Method,picard
Fit parameters,ortho=Falseextended=Truemax_iter=500
Fit,86 iterations on raw data (151506 samples)
ICA components,59
Available PCA components,60
Channel types,eeg
ICA components marked for exclusion,—


After the ICA ran, we use our recordings from the physical EOG channels aswell as the `mne_icalabel` package to classify the extracted ICs: https://mne.tools/mne-icalabel/dev/generated/examples/00_iclabel.html   
This machine learning classifier assigns a probability that an IC belongs to one of several classes: e.g., brain, muscle, eye, heart, line noise, channel noise, other.   
The excluded ICs and their properties are saved as plots to the subject's derivatives folder and can be inspected there. If we find ICs, that the algorithm would exclude, but we disagree, than we can override its decision in the cell for Manual Adjustments.

In [59]:
# ==========================================
# 7a. EOG DETECTION + ICLABEL CLASSIFICATION
# ==========================================

# We're creating quite a few plots and I want them all to pop up as interactive windows
matplotlib.use('Agg')  # Save figures to disk

eog_indices, eog_scores = ica.find_bads_eog(raw_ica) # Correlates ICs with EOG channel signals

ic_labels = label_components(raw_ica, ica, method='iclabel')
labels = ic_labels['labels']
probs = ic_labels['y_pred_proba']

# Auto-exclude: any non-brain, non-other label with prob > 0.90, plus EOG-detected
auto_exclude = sorted(set(eog_indices) | {
    idx for idx, (label, prob) in enumerate(zip(labels, probs))
    if label not in ('brain', 'other') and prob > 0.90
})

print(f"\n--- Auto-excluded ICs (>90% confidence or EOG-detected) ---")
for idx in auto_exclude:
    print(f"IC{idx:02d}: {labels[idx]} (p={probs[idx]:.2f})")

confirmed_exclude_icas = auto_exclude

# ==========================================
# 7b. SAVE DIAGNOSTIC PLOTS TO DISK
# ==========================================
subj_deriv_dir_ica = derivatives_dir / f"sub-{subj}" / "eeg" / "ica_qc"
subj_deriv_dir_ica.mkdir(parents=True, exist_ok=True)

# Overview of all components
fig_components = ica.plot_components(show=False)
if isinstance(fig_components, list):
    for i, f in enumerate(fig_components):
        f.savefig(os.path.join(subj_deriv_dir_ica, f"components_overview_{i}.png"), dpi=100)
        plt.close(f)
else:
    fig_components.savefig(os.path.join(subj_deriv_dir_ica, "components_overview.png"), dpi=100)
    plt.close(fig_components)

# Full diagnostic plots ONLY for excluded components
if confirmed_exclude_icas:
    figs = ica.plot_properties(raw_ica, picks=confirmed_exclude_icas, show=False)
    for idx, fig in zip(confirmed_exclude_icas, figs):
        fname = f"IC{idx:02d}_{labels[idx]}_{probs[idx]:.2f}.png"
        fig.savefig(os.path.join(subj_deriv_dir_ica, fname), dpi=100)
        plt.close(fig)

print(f"\nSaved QC plots to: {subj_deriv_dir_ica}")

Using EOG channels: HEOG, VEOG


... filtering ICA sources
Setting up band-pass filter from 1 - 10 Hz

FIR filter parameters
---------------------
Designing a two-pass forward and reverse, zero-phase, non-causal bandpass filter:
- Windowed frequency-domain design (firwin2) method
- Hann window
- Lower passband edge: 1.00
- Lower transition bandwidth: 0.50 Hz (-12 dB cutoff frequency: 0.75 Hz)
- Upper passband edge: 10.00 Hz
- Upper transition bandwidth: 0.50 Hz (-12 dB cutoff frequency: 10.25 Hz)
- Filter length: 2500 samples (10.000 s)

... filtering target
Setting up band-pass filter from 1 - 10 Hz

FIR filter parameters
---------------------
Designing a two-pass forward and reverse, zero-phase, non-causal bandpass filter:
- Windowed frequency-domain design (firwin2) method
- Hann window
- Lower passband edge: 1.00
- Lower transition bandwidth: 0.50 Hz (-12 dB cutoff frequency: 0.75 Hz)
- Upper passband edge: 10.00 Hz
- Upper transition bandwidth: 0.50 Hz (-12 dB cutoff frequency: 10.25 Hz)
- Filter length: 2500 sam

In [60]:
# ==========================================
# 7b. MANUAL ADJUSTMENTS
# ==========================================
# Add components here that the algorithm flagged, but you want to RESCUE !!
manual_keeps = [] 

# Combine the lists: Take the auto list, remove the keeps, and sort
confirmed_exclude_icas = [ic for ic in auto_exclude if ic not in manual_keeps]
confirmed_exclude_icas = sorted(list(set(confirmed_exclude_icas)))

print(f"\n--- Final Confirmed Exclusions ---")
print(confirmed_exclude_icas)


--- Final Confirmed Exclusions ---
[np.int64(1), np.int64(2), 6, np.int64(14), 15, 36]


In [61]:
# ==========================================
# 7d. SAVE FITTED ICA SOLUTION FOR NOTEBOOK 3
# ==========================================
# Saving the actually fitted ICA object (unmixing matrix) for Notebook 3
ica_path = subj_deriv_dir_ica / f"sub-{subj}_ica.fif"

# Store the confirmed exclusions on the object itself
ica.exclude = confirmed_exclude_icas

ica.save(ica_path, overwrite=True)
print(f"--> ICA solution saved to: {ica_path}")

Writing ICA solution to c:\Users\noahm\projects\loc_analysis\data\derivatives\sub-21\eeg\ica_qc\sub-21_ica.fif...
--> ICA solution saved to: c:\Users\noahm\projects\loc_analysis\data\derivatives\sub-21\eeg\ica_qc\sub-21_ica.fif


### 4. Saving inspection results

In the final cell, any corruptions of the data for the given participants can be documented as an additional row in the `config_df` dataframe, which is then saved in the `derivatives`folder as `preprocessing_config.csv`.
Once all subjects have been inspected with this notebook, `3_preprocessing.ipynb` will read out this information and perform the preprocessing steps accordingly.

In [62]:
# ==========================================
# 8. LOG YOUR MANUAL DECISIONS & METADATA
# ==========================================

# Check if the subject is in your predefined full-exclusion list
subject_is_excluded = subj in behavioral_exclusions

# A. Phase Completion Flags 
# Manually change the Boolean flags if a specific phase is missing or corrupted
my_eeg_enc_recorded = True
my_eeg_ret_recorded = True
my_beh_enc_complete = True
my_beh_ret_complete = True

# B. Bad Channels & ICAs
my_bad_channels = raw_eeg.info['bads'] if not subject_is_excluded else []

# B2. Build a short ICLabel justification string for each excluded IC
ica_reasons = ", ".join(
    f"IC{idx:02d}:{labels[idx]}({probs[idx]:.2f})"
    for idx in confirmed_exclude_icas
) if confirmed_exclude_icas else ""

# C. Targeted Notes
my_notes_exclusion = ""
my_notes_beh = ""
my_notes_eeg = ""
my_notes_ICA = f"ICA excluded: {ica_reasons}" if ica_reasons else ""

# ==========================================
# 9. FORMAT AND SAVE TO CSV
# ==========================================
bads_str = ", ".join(my_bad_channels) if my_bad_channels else ""
icas_str = ", ".join(map(str, confirmed_exclude_icas)) if confirmed_exclude_icas else ""

# Prepare the dictionary for the current subject
row_data = {
    "subject": subj, 
    "is_excluded": subject_is_excluded,
    "eeg_enc_recorded": my_eeg_enc_recorded,
    "eeg_ret_recorded": my_eeg_ret_recorded,
    "beh_enc_complete": my_beh_enc_complete,
    "beh_ret_complete": my_beh_ret_complete,
    "bad_channels": bads_str, 
    "bad_icas": icas_str,
    "ica_notes": my_notes_ICA,
    "notes_exclusion": my_notes_exclusion,
    "notes_beh": my_notes_beh,
    "notes_eeg": my_notes_eeg
}

if subj in df_config['subject'].values:
    # Update the existing row using the index
    idx = df_config.index[df_config['subject'] == subj].tolist()[0]
    for key, value in row_data.items():
        df_config.loc[idx, key] = value
    print(f"--> Metadata for sub-{subj} successfully updated.")
else:
    # Append a new row
    new_row = pd.DataFrame([row_data])
    df_config = pd.concat([df_config, new_row], ignore_index=True)
    print(f"--> New metadata entry for sub-{subj} successfully added.")

df_config.to_csv(config_path, index=False)
display(df_config.tail())

--> New metadata entry for sub-21 successfully added.


,subject,is_excluded,eeg_enc_recorded,eeg_ret_recorded,beh_enc_complete,beh_ret_complete,bad_channels,bad_icas,notes_exclusion,notes_beh,notes_eeg,ica_notes
0,17,False,True,True,True,True,P2,"0, 1, 2, 6, 7, 8, 10, 14, 19, 22, 23, 25, 26, ...",NaN,NaN,NaN,"ICA excluded: IC00:eye blink(1.00), IC01:eye b..."
1,20,False,True,True,True,True,"FT7, Fp1, AF8","0, 1, 2, 12, 17, 22",NaN,NaN,NaN,"ICA excluded: IC00:eye blink(1.00), IC01:eye b..."
2,21,False,True,True,True,True,"FT7, Fp1, AF8","1, 2, 6, 14, 15, 36",,,,"ICA excluded: IC01:eye blink(0.99), IC02:eye b..."
